# Theory results from L=3 networks where W2, W3 are frozen

In [7]:
import matplotlib.pyplot as plt
import numpy as np
import glob, torch, pickle, utils, model, copy, warnings
from tqdm import trange

%load_ext autoreload
%autoreload 2

parser = utils.Args('1D Gabor')
parser.add('nonlinearity', 'relu')
parser.add('loss', 'MSE') # MSE or BCE

# model parameters
parser.add('N', 1000); parser.add('Nhid', 1000); parser.add('n_layers', 3)

# task parameters
parser.add('sig_w', 0.2); parser.add('sig_s', 0.2); parser.add('theta', np.pi)
parser.add('noise_var', 0.01)

# training parameter
parser.add('eta', 1e-4)
parser.add('n_learn', 2000000)
parser.add('n_train_trials', 500)
parser.add('n_test_trials', 10000) 
parser.add('test_interval', 500)

args = parser.parse_args()
standard_stim = utils.GaborStimuli(args, simple_mode=False)
standard_net = model.Model(args)

#%% Solve self-consistent equations to get dela, delW
k = 1  # controls relative strength of L2 constraint on W and a. k=1 means equal strength.
def self_consistent(u, v, W1, a, k, x1):
    Lambda = np.linalg.inv(u*np.eye(W1.shape[1]) + (k - v)**-1 * W1.T @ W1) @ (x1 - (1-v/k)**-1 * W1.T @ a)
    Dela = (k - v)**-1 * (W1 @ Lambda + v * a)
    return np.linalg.norm(a+Dela)**2, np.linalg.norm(Lambda)**2, Lambda, Dela


update_coef = 0.9  # How quickly to update order parameters in the solver. 
#Should be between \geq 0 and < 1. Larger value means slower updates. 

sig_w_array = np.linspace(0.1, 1.0, 30) # range of sig_w to solve for

probe_args = copy.copy(args)

delw_norm = np.zeros_like(sig_w_array)

active_inds_list = []
a_list = []
delW1_list = []
for i in trange(len(sig_w_array)):
    probe_args.sig_w = sig_w_array[i]; probe_net = model.Model(probe_args)
    
    W_effs, active_inds = utils.get_effective_weights(probe_net, standard_stim.x0, full_mat=True)
    W1 = W_effs[0]
    
    full_F = W_effs[2] @ W_effs[1] @ W_effs[0]
    real_a = utils.mse_optimal_a(full_F, standard_stim, sing_val_truncation=1)
    a = W_effs[1].T @ W_effs[2].T @ real_a
    V = W_effs[0].T @ a
    x1 = standard_stim.x1_normed.t().numpy()
    lambd = (x1 - V) / np.linalg.norm(a)**2
    delw_norm[i] = np.linalg.norm(lambd) * np.linalg.norm(a)

    active_inds_list.append(active_inds)
    a_list.append(a)
    delW1_list.append(lambd.reshape(-1, 1) @ a.reshape(1, -1))






The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
calc_MLD(): MLD error is 0.15865647220940177


100%|██████████| 30/30 [00:10<00:00,  2.88it/s]


Save the results

In [9]:
file_name = 'Saved Results/theory_3L_sigsP2_frozenW2W3'
pickle.dump({'delw1':delW1_list, 'OP':None, 'a':a_list,
             'vary_values':sig_w_array, 'active_inds':active_inds_list, 'args':args, 'vary_parameter':'sigw'}, open(file_name, 'wb'))
